# Pyramidal convolution -- step-by-step test on synthetic data

Fully synthetic, no data store, no network. An exact **256x256 HEALPix
square** at level 17 centred on (lon=2.3198, lat=48.8704), with a single
Dirac at its centre.

Each step is checked before the next one runs:

1. `compute` -> `invert` == identity
2. kernel image `K = exp(-r/R0)`, `r` = distance in metres from the centre
3. pyramid of `K` -> one compact `KSZ x KSZ` kernel per band
4. convolve the data pyramid with the kernel pyramid, synthesize
5. compare: convolving a Dirac must give back `K`


## 0. Domain: an exact 256x256 square, Dirac at its centre

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))   # healpix_analyse in dev mode

import healpix_geo.nested as hgn
from healpix_analyse.decomp import HealPixDecomp
from healpix_analyse.convol import HealPixConv

LON, LAT  = 2.3198, 48.8704   # centre of the domain
LEVEL     = 17
SIDE      = 256               # square side, in pixels
JMAX      = 6                 # pyramid stages
KSZ       = 5                 # compact per-band kernel size (odd)
R0_PIX    = 10.0              # kernel scale, in finest-band pixels (see step 2)
DTYPE     = torch.float64     # float64: step 1 checks an exact identity
R_EARTH   = 6371008.8         # m

# The square is built in the base face's own (i, j) integer coordinates, so it
# is an exact 256x256 block of cells -- not a lon/lat box approximation.
centre_cell = int(np.asarray(hgn.lonlat_to_healpix([LON], [LAT], LEVEL))[0])
face, i_c, j_c = [int(v[0]) for v in hgn.healpix_to_base_cell_coordinates([centre_cell], LEVEL)]

half = SIDE // 2
ii = np.arange(i_c - half, i_c + half, dtype=np.int64)
jj = np.arange(j_c - half, j_c + half, dtype=np.int64)
JJ, II = np.meshgrid(jj, ii, indexing="ij")          # row = j, col = i
cells2d = hgn.base_cell_coordinates_to_healpix(
    np.full(II.size, face), II.ravel(), JJ.ravel(), LEVEL
).astype(np.int64).reshape(II.shape)

order = np.argsort(cells2d.ravel())
cell_ids = cells2d.ravel()[order]                    # sorted, as HealPixDecomp wants
dirac_cell = int(cells2d[half, half])                # exact centre of the square
dirac_idx = int(np.searchsorted(cell_ids, dirac_cell))


def to_image(v):
    """1D array over `cell_ids` -> the 256x256 image (row=j, col=i)."""
    img = np.full(cell_ids.size, np.nan, dtype=np.float64)
    img[order] = np.asarray(v).reshape(-1)
    return img.reshape(SIDE, SIDE)


alpha_rad = np.sqrt(4.0 * np.pi / (12.0 * (2 ** LEVEL) ** 2))   # angular pixel spacing
PIX_M = alpha_rad * R_EARTH
print(f"level {LEVEL}: pixel = {PIX_M:.1f} m, domain = {SIDE * PIX_M / 1000:.1f} x "
      f"{SIDE * PIX_M / 1000:.1f} km, {cell_ids.size} cells")
print(f"centre cell {dirac_cell} at index {dirac_idx}, face {face}, (i,j)=({i_c},{j_c})")

x = np.zeros(cell_ids.size)
x[dirac_idx] = 1.0


## 1. Test 1 -- `compute` then `invert` must be the identity

`HealPixDecomp` is a Laplacian pyramid: `detail[j] = coarse[j] - Up(coarse[j+1])`,
so synthesis is exact by construction (`S W = I`). This checks it numerically
on the Dirac *and* on a random field (a Dirac alone is a weak test).


In [ ]:
t0 = time.time()
decomp = HealPixDecomp(level=LEVEL, cell_ids=cell_ids, Jmax=JMAX,
                        ellipsoid="sphere", dtype=DTYPE)
print(f"decomp built in {time.time() - t0:.1f}s")
print("band sizes (fine -> coarse):", decomp.sizes)

pyr = decomp.compute(torch.as_tensor(x, dtype=DTYPE))
x_rec = np.asarray(decomp.invert(pyr.bands)).reshape(-1)
err_dirac = np.abs(x_rec - x).max()

rng = np.random.default_rng(0)
z = rng.standard_normal(cell_ids.size)
z_rec = np.asarray(decomp.invert(decomp.compute(torch.as_tensor(z, dtype=DTYPE)).bands)).reshape(-1)
err_rand = np.abs(z_rec - z).max()

print(f"max|invert(compute(dirac)) - dirac|  = {err_dirac:.3e}")
print(f"max|invert(compute(random)) - random| = {err_rand:.3e}")
assert err_dirac < 1e-10 and err_rand < 1e-10, "pyramid synthesis is NOT the inverse of analysis"
print("OK -- synthesis is the exact inverse of analysis")

fig, axes = plt.subplots(1, 3, figsize=(13, 4), layout="constrained")
for ax, img, ttl in ((axes[0], to_image(x), "input (Dirac)"),
                      (axes[1], to_image(x_rec), "invert(compute(input))"),
                      (axes[2], to_image(x_rec - x), "difference")):
    m = ax.imshow(img, origin="lower", cmap="viridis")
    ax.set_title(ttl, fontsize=9)
    fig.colorbar(m, ax=ax, shrink=0.8)
plt.show()


## 2. The kernel image: `K = exp(-r / R0)`, `r` in metres

`r` is the great-circle distance in metres from the domain centre (the same
cell the Dirac sits on, so the kernel and the Dirac are exactly co-located).

`R0` is set in pixels here only so the kernel's width relative to the grid is
obvious; what goes into the formula is metres. `exp(-r)` with `r` in raw
metres would decay to nothing within one pixel (49.7 m), hence the scale.


In [ ]:
lon_all, lat_all = [np.asarray(v) for v in hgn.healpix_to_lonlat(cell_ids.tolist(), LEVEL)]
lon_c, lat_c = [float(v[0]) for v in hgn.healpix_to_lonlat([dirac_cell], LEVEL)]

l1, p1 = np.radians(lon_all), np.radians(lat_all)
l2, p2 = np.radians(lon_c), np.radians(lat_c)
hav = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin((l2 - l1) / 2) ** 2
r_m = 2 * np.arcsin(np.sqrt(np.clip(hav, 0.0, 1.0))) * R_EARTH    # metres from the centre

R0_M = R0_PIX * PIX_M
K = np.exp(-r_m / R0_M)
print(f"R0 = {R0_PIX} pixels = {R0_M:.0f} m; K: max {K.max():.4g}, min {K.min():.3e}, sum {K.sum():.4g}")
print(f"K falls to 1/e at {R0_M:.0f} m, to 1% at {R0_M * np.log(100):.0f} m "
      f"({R0_PIX * np.log(100):.0f} pixels)")

K_img = to_image(K)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
m = axes[0].imshow(K_img, origin="lower", cmap="viridis")
axes[0].set_title(f"kernel K = exp(-r/{R0_M:.0f} m)", fontsize=9)
fig.colorbar(m, ax=axes[0], shrink=0.8)
axes[1].plot(np.arange(SIDE) - half, K_img[half, :], label="K, horizontal cut")
axes[1].plot(np.arange(SIDE) - half, K_img[:, half], "--", label="K, vertical cut")
axes[1].set_xlabel("pixels from centre")
axes[1].set_yscale("log")
axes[1].legend(fontsize=8)
axes[1].set_title("kernel profile (log scale)", fontsize=9)
plt.show()


## 3. Pyramid of the kernel -> one compact kernel per band

`band_j(K)` is the kernel's own content at scale `j`. What we want at band
`j` is the compact `KSZ x KSZ` stencil `w_j` such that

    band_j(dirac) (*) w_j  ==  band_j(K)

Note this is **not** "sample `band_j(K)` at 5x5 around the centre": the input
to band `j` is `band_j(dirac)`, which is a small Laplacian bump, not a Dirac,
so convolving it by `band_j(K)`'s own values does not give `band_j(K)` back
(that variant is measured at the end of this cell -- it is ~99% wrong).

`w_j` is therefore *fitted*: `KSZ**2` unknowns per band, least squares against
`band_j(K)`. The design matrix column for tap `p` is `band_j(dirac)` convolved
with a one-hot kernel at `p` -- i.e. probed through the real `HealPixConv`, so
the stencil geometry used to fit is exactly the one used to apply.

If every band's fit were exact, the synthesized result would be `K` exactly,
since `S W = I`. The per-band residuals printed below are therefore the whole
error budget of this method.


In [ ]:
pyr_K = decomp.compute(torch.as_tensor(K, dtype=DTYPE))

P = KSZ * KSZ
eye = np.eye(P)
convs, residuals, w_bands = [], [], []

t0 = time.time()
for j in range(decomp.n_bands):
    ids_j = decomp.cell_ids_per_scale[j]
    conv = HealPixConv(level=decomp.levels[j], in_channels=1, out_channels=1, kernel_sz=KSZ,
                        n_gauges=1, gauge_type="phi", cell_ids=ids_j,
                        ellipsoid="sphere", dtype=DTYPE)
    b_j = pyr.bands[j]                                  # band j of the Dirac
    k_j = np.asarray(pyr_K.bands[j]).reshape(-1)        # band j of the kernel

    A = np.zeros((k_j.size, P))                         # A[:, p] = b_j convolved with tap p alone
    for p in range(P):
        conv.set_kernel(eye[p][None, None, :], requires_grad=False)
        A[:, p] = np.asarray(conv(b_j)).reshape(-1)

    w, *_ = np.linalg.lstsq(A.T @ A + 1e-12 * np.eye(P), A.T @ k_j, rcond=None)
    res = np.linalg.norm(A @ w - k_j) / max(np.linalg.norm(k_j), 1e-30)

    conv.set_kernel(w[None, None, :], requires_grad=False)
    convs.append(conv); residuals.append(res); w_bands.append(w)
    print(f"  band {j} (level {decomp.levels[j]:2d}, {len(ids_j):6d} cells): "
          f"fit residual {res:.3f}, |w|max {np.abs(w).max():.4g}")
print(f"kernel pyramid fitted in {time.time() - t0:.1f}s")

fig, axes = plt.subplots(1, decomp.n_bands, figsize=(2.0 * decomp.n_bands, 2.4), layout="constrained")
for j, ax in enumerate(np.atleast_1d(axes)):
    ax.imshow(w_bands[j].reshape(KSZ, KSZ), cmap="RdBu_r",
              vmin=-np.abs(w_bands[j]).max(), vmax=np.abs(w_bands[j]).max())
    ax.set_title(f"w[{j}]", fontsize=8); ax.set_xticks([]); ax.set_yticks([])
plt.show()


## 4. Convolve the data pyramid with the kernel pyramid

In [ ]:
filtered = [
    torch.as_tensor(np.asarray(convs[j](pyr.bands[j])).reshape(-1), dtype=DTYPE)
    for j in range(decomp.n_bands)
]
y = np.asarray(decomp.invert(filtered)).reshape(-1)
y_img = to_image(y)
print(f"y: max {y.max():.4g}, min {y.min():.3e}, sum {y.sum():.4g}")
print(f"K: max {K.max():.4g}, min {K.min():.3e}, sum {K.sum():.4g}")

fig, ax = plt.subplots(figsize=(5, 4.2), layout="constrained")
m = ax.imshow(y_img, origin="lower", cmap="viridis")
ax.set_title(f"Dirac convolved through the kernel pyramid (Jmax={JMAX}, {KSZ}x{KSZ})", fontsize=9)
fig.colorbar(m, ax=ax, shrink=0.8)
plt.show()


## 5. Comparison: convolving a Dirac must give back the kernel

**The images below look elliptical -- that is the display, not the kernel.**
`K` is `exp(-r/R0)` with `r` a true great-circle distance, so it is exactly
circular *on the sphere*. The image axes are the base face's own integer
`(i, j)` coordinates, which at this latitude (48.87 deg, inside the HEALPix
polar cap) are neither orthogonal nor equally scaled, so a circle on the
sphere renders as a sheared ellipse. The comparison itself is done cell by
cell, so it is unaffected; the radial profile at the end of this section is
the coordinate-free version of the same check.


In [ ]:
rel = np.sqrt(np.mean((y - K) ** 2)) / np.sqrt(np.mean(K ** 2))
yn, Kn = y / y.max(), K / K.max()
rel_peak = np.sqrt(np.mean((yn - Kn) ** 2)) / np.sqrt(np.mean(Kn ** 2))
print(f"relative RMS(y - K)               : {rel:.4f}")
print(f"relative RMS, peak-normalized     : {rel_peak:.4f}")

# reference point: the finest band alone (no pyramid), fitted the same way --
# what a single compact KSZ x KSZ convolution can do against the same kernel
conv0 = convs[0]
xt = torch.as_tensor(x, dtype=DTYPE)
A0 = np.zeros((cell_ids.size, P))
for p in range(P):
    conv0.set_kernel(eye[p][None, None, :], requires_grad=False)
    A0[:, p] = np.asarray(conv0(xt)).reshape(-1)
w0, *_ = np.linalg.lstsq(A0.T @ A0 + 1e-12 * np.eye(P), A0.T @ K, rcond=None)
y_single = A0 @ w0
conv0.set_kernel(w_bands[0][None, None, :], requires_grad=False)   # restore the pyramid's own band 0

rel_single = np.sqrt(np.mean((y_single - K) ** 2)) / np.sqrt(np.mean(K ** 2))
print(f"same kernel, single {KSZ}x{KSZ} band only : {rel_single:.4f}  "
      f"(x{rel_single / max(rel, 1e-12):.1f} worse than the pyramid)")

fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
vmax = float(max(K.max(), y.max()))
m0 = axes[0].imshow(K_img, origin="lower", cmap="viridis", vmin=0, vmax=vmax)
axes[0].set_title("kernel K", fontsize=9)
m1 = axes[1].imshow(y_img, origin="lower", cmap="viridis", vmin=0, vmax=vmax)
axes[1].set_title("Dirac convolved (pyramid)", fontsize=9)
d = y_img - K_img
m2 = axes[2].imshow(d, origin="lower", cmap="RdBu_r",
                    vmin=-np.abs(d).max(), vmax=np.abs(d).max())
axes[2].set_title("difference", fontsize=9)
for ax, m in zip(axes, (m0, m1, m2)):
    fig.colorbar(m, ax=ax, shrink=0.8)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
u = np.arange(SIDE) - half
for ax, (kc, yc, ttl) in zip(axes, (
    (K_img[half, :], y_img[half, :], "horizontal cut through the centre"),
    (K_img[:, half], y_img[:, half], "vertical cut through the centre"),
)):
    ax.plot(u, kc, label="kernel K")
    ax.plot(u, yc, "--", label="Dirac convolved (pyramid)")
    ax.set_xlabel("pixels from centre"); ax.set_title(ttl, fontsize=9); ax.legend(fontsize=8)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for ax, (kc, yc, ttl) in zip(axes, (
    (K_img[half, :], y_img[half, :], "horizontal cut (log)"),
    (K_img[:, half], y_img[:, half], "vertical cut (log)"),
)):
    ax.semilogy(u, np.maximum(kc, 1e-12), label="kernel K")
    ax.semilogy(u, np.maximum(yc, 1e-12), "--", label="Dirac convolved (pyramid)")
    ax.set_xlabel("pixels from centre"); ax.set_title(ttl, fontsize=9); ax.legend(fontsize=8)
plt.show()


In [ ]:
# Coordinate-free version: both fields against the true distance in metres.
# K is isotropic by construction, so this is the comparison that actually
# matters -- unlike the i/j cuts above, it does not depend on how the face
# grid is sheared.
edges = np.linspace(0, r_m.max(), 120)
mid = 0.5 * (edges[:-1] + edges[1:])
idx = np.clip(np.digitize(r_m, edges) - 1, 0, len(mid) - 1)
count = np.bincount(idx, minlength=len(mid))
K_prof = np.bincount(idx, weights=K, minlength=len(mid)) / np.maximum(count, 1)
y_prof = np.bincount(idx, weights=y, minlength=len(mid)) / np.maximum(count, 1)
ok = count > 0

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axes[0].plot(mid[ok], K_prof[ok], label="kernel K")
axes[0].plot(mid[ok], y_prof[ok], "--", label="Dirac convolved (pyramid)")
axes[0].set_xlabel("distance from centre (m)"); axes[0].legend(fontsize=8)
axes[0].set_title("radial profile", fontsize=9)
axes[1].semilogy(mid[ok], np.maximum(K_prof[ok], 1e-12), label="kernel K")
axes[1].semilogy(mid[ok], np.maximum(y_prof[ok], 1e-12), "--", label="Dirac convolved (pyramid)")
axes[1].set_xlabel("distance from centre (m)"); axes[1].legend(fontsize=8)
axes[1].set_title("radial profile (log)", fontsize=9)
plt.show()

peak_deficit = 1.0 - y.max() / K.max()
print(f"peak: y {y.max():.4f} vs K {K.max():.4f} ({100 * peak_deficit:.1f}% low) -- the error is "
      "concentrated on the cusp of exp(-r), the hardest part for a compact stencil")
print(f"total mass: y {y.sum():.1f} vs K {K.sum():.1f} ({100 * (1 - y.sum() / K.sum()):.1f}% low)")


## What to look at

- **Step 1** must be exact (1e-15 or so). If it is not, nothing after it means
  anything.
- **Step 5**: the two images and the two cuts should overlay. Measured on this
  configuration (level 17, 256x256, `Jmax=6`, `KSZ=5`): ~12% relative RMS,
  ~7% peak-normalized, against ~x8 worse for a single `5x5` band with no
  pyramid at all.
- Widening the kernel (`R0_PIX` = 10 -> 30 -> 60) leaves that error roughly
  flat (0.122 / 0.126 / 0.109), while a single-band `5x5` cannot represent a
  60-pixel-wide kernel at all. That is the point of the pyramid: **the kernel
  stays small at every scale**.
- The per-band fit residuals printed in step 3 are the whole error budget. The
  finest bands are the worst-fitted ones; raising `KSZ` is the knob that
  targets them.
- **The far tail does not follow.** On the log radial profile, `y` tracks `K`
  down to about `1e-3` of the peak (~3 km here) and then flattens onto a floor
  of ~`1e-4`, even changing sign around 4-5 km, while `K` keeps decaying
  exponentially to `1e-11`. Below ~`1e-3` relative, what you see is the
  pyramid's own reconstruction residual, not the kernel. If an application
  needs several decades of dynamic range in the kernel's tail, this is the
  limit to know about; for a kernel used as a smoother it is irrelevant.
- On this configuration the residual is **almost entirely the cusp**: the
  profiles overlay everywhere except within a few pixels of `r = 0`, where the
  pyramid reaches ~0.85 instead of 1.0. `exp(-r)` is not differentiable at the
  origin, which is the hardest possible feature for a compact stencil -- a
  smooth kernel (Gaussian) has no such corner and fits better.
